In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

## ENV PATHS

In [ ]:
BASE_DIR_PATH = Path.cwd().parent
JDBC_DRIVER_PATH = f"{BASE_DIR_PATH}/drivers/postgresql-42.7.3.jar"
PG_HOST = ""
PG_PORT = 5432
PG_USER = ""
PG_PASSWORD = ""
PG_DATABASE = "vlr_events_metadata"
PG_TABLE = "agents"
jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"

In [ ]:
JDBC_DRIVER_PATH

In [ ]:
spark = (
    SparkSession.builder.master("local[*]")
    .config("spark.jars", JDBC_DRIVER_PATH)
    .appName("bronze-to-silver")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

## Agents Role from External DB

In [ ]:
agent_roles_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", PG_TABLE)
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
    .select("agent", "role")
)

print("agent_roles table from PostgreSQL:")
agent_roles_df.show(truncate=False)

In [ ]:
BRONZE_SCHEMA = T.StructType(
    [
        T.StructField("player_id", T.IntegerType(), nullable=False),
        T.StructField("player", T.StringType(), nullable=False),
        T.StructField("org", T.StringType(), nullable=True),
        T.StructField("agents", T.StringType(), nullable=True),
        T.StructField("rounds_played", T.IntegerType(), nullable=True),
        T.StructField("rating", T.DoubleType(), nullable=True),
        T.StructField("average_combat_score", T.DoubleType(), nullable=True),
        T.StructField("kill_deaths", T.T.T.T.DoubleType(), nullable=True),
        T.StructField("kill_assists_survived_traded", T.DoubleType(), nullable=True),
        T.StructField("average_damage_per_round", T.DoubleType(), nullable=True),
        T.StructField("kills_per_round", T.DoubleType(), nullable=True),
        T.StructField("assists_per_round", T.DoubleType(), nullable=True),
        T.StructField("first_kills_per_round", T.DoubleType(), nullable=True),
        T.StructField("first_deaths_per_round", T.DoubleType(), nullable=True),
        T.StructField("headshot_percentage", T.StringType(), nullable=True),
        T.StructField("clutch_success_percentage", T.StringType(), nullable=True),
        T.StructField("clutches_won_played_ratio", T.StringType(), nullable=True),
        T.StructField("max_kills_in_single_map", T.IntegerType(), nullable=True),
        T.StructField("kills", T.IntegerType(), nullable=True),
        T.StructField("deaths", T.IntegerType(), nullable=True),
        T.StructField("assists", T.IntegerType(), nullable=True),
        T.StructField("first_kills", T.IntegerType(), nullable=True),
        T.StructField("first_deaths", T.IntegerType(), nullable=True),
        # Partioned
        T.StructField("event_id", T.IntegerType(), nullable=False),
        T.StructField("region", T.StringType(), nullable=False),
        T.StructField("map", T.StringType(), nullable=False),
        T.StructField("agent", T.StringType(), nullable=False),
        T.StructField("snapshot_date", T.DateType(), nullable=False),
    ]
)

In [ ]:
BRONZE_DATA_PATH = BASE_DIR_PATH / "data" / "bronze"
SNAPSHOT_DATE = "2026-02-28"
snapshot_glob = (
    f"{BRONZE_DATA_PATH}/"
    f"event_id=*/region=*/map=*/agent=*/"
    f"snapshot_date={SNAPSHOT_DATE}/"
)

In [ ]:
df = (
    spark.read
    .option("header", "true")
    .option("basePath", BRONZE_DATA_PATH)
    .schema(BRONZE_SCHEMA)
    .csv(snapshot_glob)
)

print(f"Rows loaded : {df.count():,}")
print(f"Columns     : {len(df.columns)}")
df.printSchema()

In [ ]:
df.select('map', 'region', 'event_id').distinct().show(10)

In [ ]:
# Inspect raw Bronze — look at the string columns before we parse them
df.select(
    "event_id",
    "player", "agent", "region", "map",
    "kill_assists_survived_traded",
    "kills_per_round",
    "headshot_percentage",
    "clutch_success_percentage",
    "clutches_won_played_ratio"
).show(10, truncate=False)

In [ ]:
df = df.withColumnRenamed('kill_deaths', 'kill_death_ratio')

In [ ]:
for col in ["agent", "map", "region"]:
    df = df.withColumn(col, F.lower(F.trim(F.col(col))))

for col in ["player", "org"]:
    df = df.withColumn(col, F.trim(F.col(col)))

# After
print("AFTER normalization:")
df.select("agent", "map", "region", "player", "org").show(5, truncate=False)

In [ ]:
# ── Helper: parse "93%" → 0.93 ────────────────────────────────
def cast_percentage(col_name):
    """
    Strips % symbol, casts to Double, divides by 100.
    NULL input → NULL output.
    """
    return (
        F.regexp_replace(F.col(col_name), "%", "")
         .cast(T.DoubleType())
         / F.lit(100.0)
    )

# ── Helper: parse "1/2" → 0.5 ─────────────────────────────────
def cast_ratio_string(col_name):
    """
    Splits on "/" and divides numerator by denominator.
    "0/2" → 0.0  (player was in clutches but lost all — NOT a null case)
    NULL  → NULL (player was never in a clutch situation)
    denom = 0 → NULL (guard against division by zero)
    """
    numerator   = F.split(F.col(col_name), "/").getItem(0).cast(T.DoubleType())
    denominator = F.split(F.col(col_name), "/").getItem(1).cast(T.DoubleType())

    return (
        F.when(F.col(col_name).isNull(),  F.lit(None).cast(T.DoubleType()))
         .when(denominator == 0,          F.lit(None).cast(T.DoubleType()))
         .otherwise(F.round(numerator / denominator, 4))
    )

print("Helpers defined.")

In [ ]:
# ── Apply all casts ────────────────────────────────────────────

# Percentage strings → 0-1 Double
df = df.withColumn("kill_assists_survived_traded", cast_percentage("kill_assists_survived_traded"))
df = df.withColumn("headshot_percentage",          cast_percentage("headshot_percentage"))
df = df.withColumn("clutch_success_percentage",    cast_percentage("clutch_success_percentage"))

# Plain decimal strings → Double
df = df.withColumn("kills_per_round",       F.col("kills_per_round").cast(T.DoubleType()))
df = df.withColumn("assists_per_round",     F.col("assists_per_round").cast(T.DoubleType()))
df = df.withColumn("first_kills_per_round", F.col("first_kills_per_round").cast(T.DoubleType()))

# Ratio string → Double
df = df.withColumn("clutches_won_played_ratio", cast_ratio_string("clutches_won_played_ratio"))

# Re-cast columns Spark already read correctly — explicit guarantee
df = df.withColumn("player_id",                F.col("player_id").cast(T.IntegerType()))
df = df.withColumn("rounds_played",            F.col("rounds_played").cast(T.IntegerType()))
df = df.withColumn("rating",                   F.col("rating").cast(T.DoubleType()))
df = df.withColumn("average_combat_score",     F.col("average_combat_score").cast(T.DoubleType()))
df = df.withColumn("kill_death_ratio",         F.col("kill_death_ratio").cast(T.DoubleType()))
df = df.withColumn("average_damage_per_round", F.col("average_damage_per_round").cast(T.DoubleType()))
df = df.withColumn("first_deaths_per_round",   F.col("first_deaths_per_round").cast(T.DoubleType()))
df = df.withColumn("max_kills_in_single_map",  F.col("max_kills_in_single_map").cast(T.IntegerType()))
df = df.withColumn("kills",                    F.col("kills").cast(T.IntegerType()))
df = df.withColumn("deaths",                   F.col("deaths").cast(T.IntegerType()))
df = df.withColumn("assists",                  F.col("assists").cast(T.IntegerType()))
df = df.withColumn("first_kills",              F.col("first_kills").cast(T.IntegerType()))
df = df.withColumn("first_deaths",             F.col("first_deaths").cast(T.IntegerType()))
df = df.withColumn("snapshot_date",            F.col("snapshot_date").cast(T.DateType()))

print("All casts applied. Schema after casting:")
df.printSchema()

In [ ]:
# Validate — spot check the parsed values look right
df.select(
    "kill_assists_survived_traded",  # should be 0.0 - 1.0
    "headshot_percentage",           # should be 0.0 - 1.0
    "clutch_success_percentage",     # should be 0.0 - 1.0 or NULL
    "clutches_won_played_ratio",     # should be 0.0 - 1.0 or NULL
    "kills_per_round",               # should be 0.5 - 1.5 range typically
).show(10)

In [ ]:
RATIO_TOLERANCE = 0.05  # 5%


def ratio_mismatch(stored_col, numerator_col, denominator_col):
    denom = F.nullif(F.col(denominator_col), F.lit(0))
    computed = F.col(numerator_col) / denom
    stored = F.col(stored_col)
    safe_computed = F.nullif(computed, F.lit(0.0))
    deviation = F.abs(stored - safe_computed) / F.abs(safe_computed)
    return (
        stored.isNotNull() & safe_computed.isNotNull() & (deviation > RATIO_TOLERANCE)
    )


# dq_low_sample
df = df.withColumn(
    "dq_low_sample", F.col("rounds_played").isNull() | (F.col("rounds_played") < 50)
)

# dq_clutch_no_attempts — & (AND) not | (OR)
# BOTH must be null → player was never in a clutch situation
# NULL percentage + "0/2" ratio → player attempted but won none (not flagged)
df = df.withColumn(
    "dq_clutch_no_attempts",
    F.col("clutch_success_percentage").isNull()
    & F.col("clutches_won_played_ratio").isNull(),
)

# Fill clutch nulls with 0.0 AFTER flag is set
df = df.withColumn(
    "clutch_success_percentage",
    F.coalesce(F.col("clutch_success_percentage"), F.lit(0.0)),
)
df = df.withColumn(
    "clutches_won_played_ratio",
    F.coalesce(F.col("clutches_won_played_ratio"), F.lit(0.0)),
)

# dq_null_core_fields
df = df.withColumn(
    "dq_null_core_fields",
    F.col("rating").isNull()
    | F.col("average_combat_score").isNull()
    | F.col("kills").isNull(),
)

# dq_ratio_mismatch
df = df.withColumn(
    "dq_ratio_mismatch",
    ratio_mismatch("kill_death_ratio", "kills", "deaths")
    | ratio_mismatch("kills_per_round", "kills", "rounds_played")
    | ratio_mismatch("first_kills_per_round", "first_kills", "rounds_played")
    | ratio_mismatch("first_deaths_per_round", "first_deaths", "rounds_played"),
)

print("DQ flags added. Summary:")

In [ ]:
# Check what happened to clutch nulls after casting
df.select(
    "clutch_success_percentage",
    "clutches_won_played_ratio"
).filter(
    F.col("clutch_success_percentage") == 0.0
).show(10, truncate=False)

In [ ]:
# Read raw Bronze again (before any transforms) to see original values
df_raw = (
    spark.read
    .option("header", "true")
    .option("basePath", BRONZE_DATA_PATH)
    .csv(f"{BRONZE_DATA_PATH}/event_id=*/region=*/map=*/agent=*/snapshot_date={SNAPSHOT_DATE}/")
)

# Show raw clutch values including nulls, empty strings, zeros
df_raw.select(
    "clutch_success_percentage",
    "clutches_won_played_ratio"
).groupBy(
    "clutch_success_percentage",
    "clutches_won_played_ratio"
).count().orderBy(F.col("count").desc()).show(20, truncate=False)

In [ ]:
# DQ Summary — how many rows are flagged for each issue?
total = df.count()
dq_cols = [
    "dq_low_sample",
    "dq_clutch_no_attempts",
    "dq_null_core_fields",
    "dq_ratio_mismatch",
]

print(f"Total rows: {total:,}")
print("-" * 45)
for flag in dq_cols:
    count = df.filter(F.col(flag) == True).count()
    pct = (count / total * 100) if total > 0 else 0
    print(f"  {flag:<28} {count:>6,}  ({pct:.1f}%)")

In [ ]:
# fk_fd_ratio
# Edge cases:
#   first_deaths=0, first_kills>0 → perfect entry record → sentinel 99.0
#   first_deaths=0, first_kills=0 → never entry fragged at all → NULL
df = df.withColumn(
    "fk_fd_ratio",
    F.when(
        F.col("first_deaths").isNull() | F.col("first_kills").isNull(),
        F.lit(None).cast(T.DoubleType())
    ).when(
        (F.col("first_deaths") == 0) & (F.col("first_kills") > 0),
        F.lit(99.0)
    ).when(
        (F.col("first_deaths") == 0) & (F.col("first_kills") == 0),
        F.lit(None).cast(T.DoubleType())
    ).otherwise(
        F.round(F.col("first_kills") / F.col("first_deaths"), 3)
    )
)

# net_first_blood
df = df.withColumn(
    "net_first_blood",
    F.when(
        F.col("first_kills").isNotNull() & F.col("first_deaths").isNotNull(),
        F.col("first_kills") - F.col("first_deaths")
    ).otherwise(F.lit(None).cast(T.IntegerType()))
)

# damage_delta
df = df.withColumn(
    "damage_delta",
    F.when(
        F.col("average_damage_per_round").isNotNull(),
        F.round(F.col("average_damage_per_round") - 150.0, 2)
    ).otherwise(F.lit(None).cast(T.DoubleType()))
)

# Spot check the derived metrics
df.select(
    "player", "first_kills", "first_deaths",
    "fk_fd_ratio", "net_first_blood",
    "average_damage_per_round", "damage_delta"
).show(10)